# Historischer Wettervergleich Berlin 2025

Dieses Notebook vergleicht die täglichen Mitteltemperaturen des Jahres 2025 für Berlin-Tempelhof. Als Stationsmessung dient der Deutsche Wetterdienst (DWD), als Vergleich der historische Open-Meteo-Datensatz am Standort der DWD-Station.

Die Daten werden beim Ausführen live geladen. Dafür ist eine Internetverbindung erforderlich; zusätzliche Python-Pakete oder API-Schlüssel sind nicht nötig.

## Zeitraum, Ort und Vergleich

Untersucht wird der Zeitraum vom 1. Januar bis zum 31. Dezember 2025. Die DWD-Station Berlin-Tempelhof trägt die Stationsnummer 00433. Verglichen wird `TMK` (DWD-Tagesmitteltemperatur in °C) mit Open-Meteos täglichem Mittel der 2-Meter-Temperatur. Abweichungen sind zu erwarten, da eine Stationsmessung und ein modellierter/reanalysierter Gitterwert unterschiedliche Datenquellen sind.

In [ ]:
import csv
import json
from io import BytesIO
from statistics import mean, correlation
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen
from zipfile import BadZipFile, ZipFile

## 1. Historische Vergleichsdaten von Open-Meteo

Open-Meteo liefert tägliche Mitteltemperaturen für denselben Standort und Zeitraum. Diese Daten werden später mit den DWD-Tageswerten anhand des Datums zusammengeführt.

In [ ]:
parameter_open_meteo = {
    "latitude": 52.4676,
    "longitude": 13.4020,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "daily": "temperature_2m_mean",
    "timezone": "Europe/Berlin",
}
url_open_meteo = (
    "https://archive-api.open-meteo.com/v1/archive?"
    + urlencode(parameter_open_meteo)
)
anfrage_open_meteo = Request(
    url_open_meteo,
    headers={"User-Agent": "BerlinWeatherComparisonNotebook/1.0"},
)

try:
    with urlopen(anfrage_open_meteo, timeout=30) as antwort:
        open_meteo_daten = json.load(antwort)
except (HTTPError, URLError, TimeoutError, json.JSONDecodeError) as fehler:
    raise RuntimeError(
        "Open-Meteo-Daten konnten nicht geladen werden. Prüfe die Internetverbindung."
    ) from fehler

open_meteo_tagesmittel = {
    datum.replace("-", ""): temperatur
    for datum, temperatur in zip(
        open_meteo_daten["daily"]["time"],
        open_meteo_daten["daily"]["temperature_2m_mean"],
    )
    if temperatur is not None
}

print(f"Open-Meteo-Tage geladen: {len(open_meteo_tagesmittel)}")

## 2. DWD-Tageswerte herunterladen

Die DWD-CDC-Datei enthält die Stationsmessungen. Aus dem ZIP-Archiv wird die Produktdatei gelesen; `TMK` ist das tägliche Temperaturmittel in Grad Celsius. Fehlwerte (`-999`) werden ausgelassen.

In [ ]:
dwd_url = (
    "https://opendata.dwd.de/climate_environment/CDC/observations_germany/"
    "climate/daily/kl/historical/"
    "tageswerte_KL_00433_19480101_20251231_hist.zip"
)
anfrage_dwd = Request(
    dwd_url,
    headers={"User-Agent": "BerlinWeatherComparisonNotebook/1.0"},
)

try:
    with urlopen(anfrage_dwd, timeout=30) as antwort:
        dwd_zip_bytes = antwort.read()
    with ZipFile(BytesIO(dwd_zip_bytes)) as archiv:
        produktdatei = next(
            name for name in archiv.namelist()
            if name.startswith("produkt_klima_tag_")
        )
        text = archiv.read(produktdatei).decode("latin-1").splitlines()
except (HTTPError, URLError, TimeoutError, BadZipFile, StopIteration) as fehler:
    raise RuntimeError(
        "Die DWD-Tagesdatei konnte nicht geladen oder gelesen werden."
    ) from fehler

leser = csv.DictReader(text, delimiter=";")
dwd_tagesmittel = {}
for zeile in leser:
    zeile = {schluessel.strip(): wert.strip() for schluessel, wert in zeile.items()}
    datum = zeile["MESS_DATUM"]
    if datum.startswith("2025"):
        temperatur = float(zeile["TMK"])
        if temperatur > -900:
            dwd_tagesmittel[datum] = temperatur

print(f"DWD-Tage geladen: {len(dwd_tagesmittel)}")

## 3. Jahresvergleich und Abweichungen

Beide Reihen werden nach Datum zusammengeführt. Ausgewiesen werden die Jahresmittel, der mittlere absolute Fehler (MAE), die mittlere Abweichung (Open-Meteo minus DWD) und die Korrelation der Tagesmittel.

In [ ]:
gemeinsame_tage = sorted(open_meteo_tagesmittel.keys() & dwd_tagesmittel.keys())
if not gemeinsame_tage:
    raise ValueError("Für 2025 wurden keine gemeinsamen Tage gefunden.")

open_meteo_werte = [open_meteo_tagesmittel[tag] for tag in gemeinsame_tage]
dwd_werte = [dwd_tagesmittel[tag] for tag in gemeinsame_tage]
abweichungen = [open_wert - dwd_wert for open_wert, dwd_wert in zip(open_meteo_werte, dwd_werte)]

print("Berlin-Tempelhof | Tagesmitteltemperatur 2025")
print("=" * 52)
print(f"Gemeinsame Tage:                 {len(gemeinsame_tage)}")
print(f"Jahresmittel DWD:                {mean(dwd_werte):.2f} °C")
print(f"Jahresmittel Open-Meteo:         {mean(open_meteo_werte):.2f} °C")
print(f"Mittlerer absoluter Fehler:      {mean(abs(wert) for wert in abweichungen):.2f} °C")
print(f"Mittlere Abweichung (OM - DWD):  {mean(abweichungen):+.2f} °C")
if len(gemeinsame_tage) > 1:
    print(f"Korrelation der Tageswerte:      {correlation(dwd_werte, open_meteo_werte):.3f}")

print("\nMonatsmittel im Vergleich:")
for monat in sorted({tag[:6] for tag in gemeinsame_tage}):
    monatstage = [tag for tag in gemeinsame_tage if tag.startswith(monat)]
    dwd_monat = mean(dwd_tagesmittel[tag] for tag in monatstage)
    open_meteo_monat = mean(open_meteo_tagesmittel[tag] for tag in monatstage)
    print(
        f"{monat[4:]}.{monat[:4]} | DWD {dwd_monat:6.2f} °C | "
        f"Open-Meteo {open_meteo_monat:6.2f} °C | "
        f"Differenz {open_meteo_monat - dwd_monat:+.2f} °C"
    )

## Datenquellen und Interpretation

- **DWD Climate Data Center:** tägliche Stationsmessungen für Berlin-Tempelhof, Stations-ID 00433. Der Wert `TMK` ist das Tagesmittel in °C.
- **Open-Meteo Archive API:** historische tägliche 2-Meter-Mitteltemperatur an den Koordinaten der DWD-Station. Das ist ein Gitter-/Reanalysewert und keine Stationsmessung.

Die Datensätze sind daher nicht identisch. Kleine Unterschiede sind normal. Die DWD-Datei und Open-Meteo können aktualisiert werden; beim erneuten Ausführen werden die jeweils verfügbaren Werte geladen.